# CMSC 678 Project by Chris Dollo and Madeline Rippin

## Setup

Set `HOME_FOLDER` to your project path on Drive. `DATA_DIR` is where the data goes, all `.npz` and `.mat` files need to be in the same folder.

In [ ]:
HOME_FOLDER = '/content/drive/MyDrive/Masters/First Year/CMSC 678/CMSC678Project'
DATA_DIR = HOME_FOLDER + '/data'

## Mount Google Drive

In [ ]:
import os

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(DATA_DIR, exist_ok=True)
print(f'HOME_FOLDER: {HOME_FOLDER}')
print(f'DATA_DIR: {DATA_DIR}')


HOME_FOLDER: /content/drive/MyDrive/Masters/First Year/CMSC 678/CMSC678Project
DATA_DIR:    /content/drive/MyDrive/Masters/First Year/CMSC 678/CMSC678Project/data


In [ ]:
!ls /content/drive/MyDrive/Masters/'First Year'/'CMSC 678'/CMSC678Project

ablation.py	     file_dependency_diagram.png  reimplementation
ATCNet.py	     MAIN_RUN.py		  requirements.txt
data		     Mamba.py			  results
desc_2a.pdf	     normal_results.py		  sr_augmentation.py
EEGNetBYOL.py	     paper.pdf			  summary.txt
EEGNet.py	     plot.py			  Untitled0.ipynb
FBCSP_Multiclass.py  __pycache__		  utils.py
FBCSP_V4.py	     README.txt


## Install dependencies

`mamba-ssm` needs `--no-build-isolation` or it fails to compile. Everything else is in `requirements.txt`.

In [ ]:
%cd "$HOME_FOLDER"

!pip install -q -r requirements.txt
!pip install -q mamba-ssm --no-build-isolation

/content/drive/MyDrive/Masters/First Year/CMSC 678/CMSC678Project


## Sanity check

Loads subject 01 through the full pipeline to make sure shapes look right. After resampling 250 → 128 Hz we expect `(288, 22, 512)`.

In [ ]:
import numpy as np, sys

# set before importing so normal_results.py picks up the right data dir
os.environ['BCI_DATA_DIR'] = DATA_DIR
if HOME_FOLDER not in sys.path:
    sys.path.insert(0, HOME_FOLDER)

from utils import load_data, subject_to_arrays

subjectData, subjectDataEVAL = load_data(data_dir=DATA_DIR)
X_train, y_train, X_eval, y_eval = subject_to_arrays(
    subjectData['subject01'], subjectDataEVAL['subject01'],
    os.path.join(DATA_DIR, 'A01E.mat'))

print(f'X_train: {X_train.shape}   X_eval: {X_eval.shape}')
assert X_train.shape == X_eval.shape == (288, 22, 512)
assert set(np.unique(y_train)) == set(np.unique(y_eval)) == {0, 1, 2, 3}
print('shapes look good, pipeline is working')

---
## Experiment 1: FBCSP baseline

Traditional signal processing baseline. Also defines the high/low performer split (median FBCSP accuracy). Should get ~0.634 mean.

Results cached to `fbcsp.pkl`. Flip `FORCE_RERUN_FBCSP = True` to retrain.

In [ ]:
import pickle, time

FORCE_RERUN_FBCSP = False

RESULTS_DIR = os.path.join(HOME_FOLDER, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)
fbcsp_cache = os.path.join(RESULTS_DIR, 'fbcsp.pkl')

if os.path.exists(fbcsp_cache) and not FORCE_RERUN_FBCSP:
    with open(fbcsp_cache, 'rb') as f:
        loaded = pickle.load(f)
    fbcsp_acc, fbcsp_cm = loaded[0], loaded[1]
    fbcsp_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {fbcsp_runtime/60:.1f} min' if fbcsp_runtime else 'prior runtime not recorded'
    print(f'Loaded cached FBCSP results ({msg})')
else:
    print('Running FBCSP across all 9 subjects...')
    from normal_results import FBCSP_results
    t0 = time.perf_counter()
    fbcsp_acc, _, fbcsp_cm = FBCSP_results()
    fbcsp_runtime = time.perf_counter() - t0
    with open(fbcsp_cache, 'wb') as f:
        pickle.dump((fbcsp_acc, fbcsp_cm, fbcsp_runtime), f)
    print(f'Wall time: {fbcsp_runtime/60:.1f} min   cached -> {fbcsp_cache}')

fbcsp_acc = np.array(fbcsp_acc)

print('\nPer-subject accuracy:')
for i, a in enumerate(fbcsp_acc, 1):
    print(f'  S{i:02d}: {a:.4f}')
print(f'  mean: {fbcsp_acc.mean():.4f}   median: {np.median(fbcsp_acc):.4f}   (paper mean: 0.634)')

# define high/low performer groups
median = np.median(fbcsp_acc)
high = [i for i, a in enumerate(fbcsp_acc, 1) if a >= median]
low  = [i for i, a in enumerate(fbcsp_acc, 1) if a <  median]
print(f'\nHigh (>= median): {high}    paper: [1, 3, 7, 8, 9]')
print(f'Low  (<  median): {low}    paper: [2, 4, 5, 6]')

# aggregate CM across all subjects
print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], fbcsp_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

Loaded cached FBCSP results (prior runtime not recorded)

Per-subject accuracy:
  S01: 0.6944
  S02: 0.5347
  S03: 0.7986
  S04: 0.5868
  S05: 0.5208
  S06: 0.4062
  S07: 0.7847
  S08: 0.7118
  S09: 0.6944
  mean: 0.6370   median: 0.6944   (paper mean: 0.634)

High (>= median): [1, 3, 7, 8, 9]    paper: [1, 3, 7, 8, 9]
Low  (<  median): [2, 4, 5, 6]    paper: [2, 4, 5, 6]

Confusion matrix (rows=true, cols=pred):
              Left   Right  Feet   Tongue
  true Left       426    130     41     51
  true Right       82    471     66     29
  true Feet        91     62    367    128
  true Tongue      89     92     80    387


---
## Experiment 2: EEGNet (no augmentation)

Smallest deep CNN built for EEG. Does deep learning beat FBCSP, and does the gain favor low performers?

300 epochs per subject. Cached to `eegnet_noaug.pkl`.

In [ ]:
#flip to True to ignore the pickle cache and retrain EEGNet.
FORCE_RERUN_EEGNET = False

eegnet_cache = os.path.join(RESULTS_DIR, 'eegnet_noaug.pkl')

if os.path.exists(eegnet_cache) and not FORCE_RERUN_EEGNET:
    with open(eegnet_cache, 'rb') as f:
        loaded = pickle.load(f)
    eegnet_acc, eegnet_cm = loaded[0], loaded[1]
    eegnet_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {eegnet_runtime/60:.1f} min' if eegnet_runtime else 'prior runtime not recorded'
    print(f'Loaded cached EEGNet results ({msg})')
else:
    print('Training EEGNet on all 9 subjects (300 epochs each)...')
    from normal_results import EEG_results
    t0 = time.perf_counter()
    # no augmentation
    eegnet_acc, _, eegnet_cm = EEG_results(aug_bool=False)
    eegnet_runtime = time.perf_counter() - t0
    with open(eegnet_cache, 'wb') as f:
        pickle.dump((eegnet_acc, eegnet_cm, eegnet_runtime), f)
    print(f'Wall time: {eegnet_runtime/60:.1f} min   cached -> {eegnet_cache}')

eegnet_acc = np.array(eegnet_acc)

#per-subject comparison against FBCSP
print('\nPer-subject accuracy (EEGNet vs FBCSP):')
for i, (a_eeg, a_fb) in enumerate(zip(eegnet_acc, fbcsp_acc), 1):
    delta = a_eeg - a_fb
    print(f'  S{i:02d}: EEGNet {a_eeg:.4f}   FBCSP {a_fb:.4f}   Δ {delta:+.4f}')
print(f'  mean: EEGNet {eegnet_acc.mean():.4f}   FBCSP {fbcsp_acc.mean():.4f}   Δ {eegnet_acc.mean()-fbcsp_acc.mean():+.4f}')

#does the gain over FBCSP favor low performers?
high_idx = [i-1 for i in [1, 3, 7, 8, 9]]
low_idx  = [i-1 for i in [2, 4, 5, 6]]
gain_high = (eegnet_acc[high_idx] - fbcsp_acc[high_idx]).mean()
gain_low  = (eegnet_acc[low_idx]  - fbcsp_acc[low_idx]).mean()
print(f'\nMean Δ over FBCSP, by group:')
print(f'  high performers {[1,3,7,8,9]}: {gain_high:+.4f}')
print(f'  low  performers {[2,4,5,6]}  : {gain_low:+.4f}')
verdict = 'low performers benefit more' if gain_low > gain_high else 'high performers benefit more'
print(f'  -> low minus high: {gain_low - gain_high:+.4f}   ({verdict})')

#CM
print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], eegnet_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

Loaded cached EEGNet results (prior run took 2.0 min)

Per-subject accuracy (EEGNet vs FBCSP):
  S01: EEGNet 0.8160   FBCSP 0.6944   Δ +0.1215
  S02: EEGNet 0.5556   FBCSP 0.5347   Δ +0.0208
  S03: EEGNet 0.8750   FBCSP 0.7986   Δ +0.0764
  S04: EEGNet 0.6215   FBCSP 0.5868   Δ +0.0347
  S05: EEGNet 0.6493   FBCSP 0.5208   Δ +0.1285
  S06: EEGNet 0.5625   FBCSP 0.4062   Δ +0.1562
  S07: EEGNet 0.6979   FBCSP 0.7847   Δ -0.0868
  S08: EEGNet 0.8056   FBCSP 0.7118   Δ +0.0938
  S09: EEGNet 0.7778   FBCSP 0.6944   Δ +0.0833
  mean: EEGNet 0.7068   FBCSP 0.6370   Δ +0.0698

Mean Δ over FBCSP, by group:
  high performers [1, 3, 7, 8, 9]: +0.0576
  low  performers [2, 4, 5, 6]  : +0.0851
  -> low minus high: +0.0274   (LOW benefit MORE — supports BCI-inefficiency hypothesis)

Confusion matrix (rows=true, cols=pred):
              Left   Right  Feet   Tongue
  true Left       455     69     71     53
  true Right       61    461     75     51
  true Feet        41     54    470     83
  true 

---
## Experiment 3: ATCNet (no augmentation)

ATCNet adds sliding-window attention on top of the EEGNet front-end. Does more capacity and longer temporal context actually help?

300 epochs per subject. Cached to `atcnet_noaug.pkl`.

In [ ]:
#flip to True to ignore the pickle cache and retrain ATCNet.
FORCE_RERUN_ATCNET = False

atcnet_cache = os.path.join(RESULTS_DIR, 'atcnet_noaug.pkl')

if os.path.exists(atcnet_cache) and not FORCE_RERUN_ATCNET:
    with open(atcnet_cache, 'rb') as f:
        loaded = pickle.load(f)
    atcnet_acc, atcnet_cm = loaded[0], loaded[1]
    atcnet_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {atcnet_runtime/60:.1f} min' if atcnet_runtime else 'prior runtime not recorded'
    print(f'Loaded cached ATCNet results ({msg})')
else:
    print('Training ATCNet on all 9 subjects (300 epochs each)...')
    from normal_results import ATCNet_results
    t0 = time.perf_counter()
    # no augmentation
    atcnet_acc, _, atcnet_cm = ATCNet_results(aug_bool=False)
    atcnet_runtime = time.perf_counter() - t0
    with open(atcnet_cache, 'wb') as f:
        pickle.dump((atcnet_acc, atcnet_cm, atcnet_runtime), f)
    print(f'Wall time: {atcnet_runtime/60:.1f} min   cached -> {atcnet_cache}')

atcnet_acc = np.array(atcnet_acc)

#per-subject comparison against FBCSP and EEGNet
print('\nPer-subject accuracy (ATCNet vs EEGNet vs FBCSP):')
for i, (a_atc, a_eeg, a_fb) in enumerate(zip(atcnet_acc, eegnet_acc, fbcsp_acc), 1):
    print(f'  S{i:02d}: ATCNet {a_atc:.4f}   EEGNet {a_eeg:.4f}   FBCSP {a_fb:.4f}   '
          f'Δ_FBCSP {a_atc-a_fb:+.4f}   Δ_EEGNet {a_atc-a_eeg:+.4f}')
print(f'  mean: ATCNet {atcnet_acc.mean():.4f}   EEGNet {eegnet_acc.mean():.4f}   '
      f'FBCSP {fbcsp_acc.mean():.4f}')

#does the gain over FBCSP favor low performers?
high_idx = [i-1 for i in [1, 3, 7, 8, 9]]
low_idx  = [i-1 for i in [2, 4, 5, 6]]
gain_high = (atcnet_acc[high_idx] - fbcsp_acc[high_idx]).mean()
gain_low  = (atcnet_acc[low_idx]  - fbcsp_acc[low_idx]).mean()
print(f'\nMean Δ over FBCSP, by group:')
print(f'  high performers {[1,3,7,8,9]}: {gain_high:+.4f}')
print(f'  low  performers {[2,4,5,6]}  : {gain_low:+.4f}')
verdict = 'low performers benefit more' if gain_low > gain_high else 'high performers benefit more'
print(f'  -> low minus high: {gain_low - gain_high:+.4f}   ({verdict})')

#CM
print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], atcnet_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

Loaded cached ATCNet results (prior run took 10.6 min)

Per-subject accuracy (ATCNet vs EEGNet vs FBCSP):
  S01: ATCNet 0.7188   EEGNet 0.8160   FBCSP 0.6944   Δ_FBCSP +0.0243   Δ_EEGNet -0.0972
  S02: ATCNet 0.5417   EEGNet 0.5556   FBCSP 0.5347   Δ_FBCSP +0.0069   Δ_EEGNet -0.0139
  S03: ATCNet 0.8438   EEGNet 0.8750   FBCSP 0.7986   Δ_FBCSP +0.0451   Δ_EEGNet -0.0312
  S04: ATCNet 0.6493   EEGNet 0.6215   FBCSP 0.5868   Δ_FBCSP +0.0625   Δ_EEGNet +0.0278
  S05: ATCNet 0.6806   EEGNet 0.6493   FBCSP 0.5208   Δ_FBCSP +0.1597   Δ_EEGNet +0.0312
  S06: ATCNet 0.5868   EEGNet 0.5625   FBCSP 0.4062   Δ_FBCSP +0.1806   Δ_EEGNet +0.0243
  S07: ATCNet 0.6736   EEGNet 0.6979   FBCSP 0.7847   Δ_FBCSP -0.1111   Δ_EEGNet -0.0243
  S08: ATCNet 0.7500   EEGNet 0.8056   FBCSP 0.7118   Δ_FBCSP +0.0382   Δ_EEGNet -0.0556
  S09: ATCNet 0.7465   EEGNet 0.7778   FBCSP 0.6944   Δ_FBCSP +0.0521   Δ_EEGNet -0.0312
  mean: ATCNet 0.6879   EEGNet 0.7068   FBCSP 0.6370

Mean Δ over FBCSP, by group:
  high per

---
## Experiment 4: MI-Mamba (no augmentation)

Mamba (selective state-space model) as an alternative to attention. Runs 500 epochs since it needs longer to converge.

Cached to `mamba_noaug.pkl`.

In [ ]:
#flip to True to ignore the pickle cache and retrain Mamba.
FORCE_RERUN_MAMBA = False

mamba_cache = os.path.join(RESULTS_DIR, 'mamba_noaug.pkl')

if os.path.exists(mamba_cache) and not FORCE_RERUN_MAMBA:
    with open(mamba_cache, 'rb') as f:
        loaded = pickle.load(f)
    mamba_acc, mamba_cm = loaded[0], loaded[1]
    mamba_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {mamba_runtime/60:.1f} min' if mamba_runtime else 'prior runtime not recorded'
    print(f'Loaded cached Mamba results ({msg})')
else:
    print('Training MI-Mamba on all 9 subjects (500 epochs each)...')
    from normal_results import Mamba_results
    t0 = time.perf_counter()
    # no augmentation
    mamba_acc, _, mamba_cm = Mamba_results(aug_bool=False)
    mamba_runtime = time.perf_counter() - t0
    with open(mamba_cache, 'wb') as f:
        pickle.dump((mamba_acc, mamba_cm, mamba_runtime), f)
    print(f'Wall time: {mamba_runtime/60:.1f} min   cached -> {mamba_cache}')

mamba_acc = np.array(mamba_acc)

#per-subject comparison against FBCSP and EEGNet and ATCNet
print('\nPer-subject accuracy (Mamba vs ATCNet vs EEGNet vs FBCSP):')
for i, (a_m, a_atc, a_eeg, a_fb) in enumerate(zip(mamba_acc, atcnet_acc, eegnet_acc, fbcsp_acc), 1):
    print(f'  S{i:02d}: Mamba {a_m:.4f}   ATCNet {a_atc:.4f}   EEGNet {a_eeg:.4f}   FBCSP {a_fb:.4f}   '
          f'Δ_FBCSP {a_m-a_fb:+.4f}   Δ_ATCNet {a_m-a_atc:+.4f}')
print(f'  mean: Mamba {mamba_acc.mean():.4f}   ATCNet {atcnet_acc.mean():.4f}   '
      f'EEGNet {eegnet_acc.mean():.4f}   FBCSP {fbcsp_acc.mean():.4f}')

#does the gain over FBCSP favor low performers?
high_idx = [i-1 for i in [1, 3, 7, 8, 9]]
low_idx  = [i-1 for i in [2, 4, 5, 6]]
gain_high = (mamba_acc[high_idx] - fbcsp_acc[high_idx]).mean()
gain_low  = (mamba_acc[low_idx]  - fbcsp_acc[low_idx]).mean()
print(f'\nMean Δ over FBCSP, by group:')
print(f'  high performers {[1,3,7,8,9]}: {gain_high:+.4f}')
print(f'  low  performers {[2,4,5,6]}  : {gain_low:+.4f}')
verdict = 'low performers benefit more' if gain_low > gain_high else 'high performers benefit more'
print(f'  -> low minus high: {gain_low - gain_high:+.4f}   ({verdict})')

#CM
print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], mamba_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

Loaded cached Mamba results (prior run took 3.3 min)

Per-subject accuracy (Mamba vs ATCNet vs EEGNet vs FBCSP):
  S01: Mamba 0.6007   ATCNet 0.7188   EEGNet 0.8160   FBCSP 0.6944   Δ_FBCSP -0.0938   Δ_ATCNet -0.1181
  S02: Mamba 0.3611   ATCNet 0.5417   EEGNet 0.5556   FBCSP 0.5347   Δ_FBCSP -0.1736   Δ_ATCNet -0.1806
  S03: Mamba 0.7326   ATCNet 0.8438   EEGNet 0.8750   FBCSP 0.7986   Δ_FBCSP -0.0660   Δ_ATCNet -0.1111
  S04: Mamba 0.4375   ATCNet 0.6493   EEGNet 0.6215   FBCSP 0.5868   Δ_FBCSP -0.1493   Δ_ATCNet -0.2118
  S05: Mamba 0.2847   ATCNet 0.6806   EEGNet 0.6493   FBCSP 0.5208   Δ_FBCSP -0.2361   Δ_ATCNet -0.3958
  S06: Mamba 0.3472   ATCNet 0.5868   EEGNet 0.5625   FBCSP 0.4062   Δ_FBCSP -0.0590   Δ_ATCNet -0.2396
  S07: Mamba 0.5069   ATCNet 0.6736   EEGNet 0.6979   FBCSP 0.7847   Δ_FBCSP -0.2778   Δ_ATCNet -0.1667
  S08: Mamba 0.6736   ATCNet 0.7500   EEGNet 0.8056   FBCSP 0.7118   Δ_FBCSP -0.0382   Δ_ATCNet -0.0764
  S09: Mamba 0.6250   ATCNet 0.7465   EEGNet 0.7778   F

---
## Experiment 5: EEGNet + S&R augmentation

S&R (Segmentation & Recombination) triples the training set by splitting trials in half and swapping halves across same-class donors. 288 → 864 trials. Does more data close the gap for low performers?

Cached to `eegnet_aug.pkl`.

In [ ]:
#flip to True to ignore the pickle cache and retrain EEGNet+S&R.
FORCE_RERUN_EEGNET_AUG = False

eegnet_aug_cache = os.path.join(RESULTS_DIR, 'eegnet_aug.pkl')

if os.path.exists(eegnet_aug_cache) and not FORCE_RERUN_EEGNET_AUG:
    with open(eegnet_aug_cache, 'rb') as f:
        loaded = pickle.load(f)
    eegnet_aug_acc, eegnet_aug_cm = loaded[0], loaded[1]
    eegnet_aug_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {eegnet_aug_runtime/60:.1f} min' if eegnet_aug_runtime else 'prior runtime not recorded'
    print(f'Loaded cached EEGNet+S&R results ({msg})')
else:
    print('Training EEGNet+S&R on all 9 subjects (300 epochs each, 864 trials/subject)...')
    from normal_results import EEG_results
    t0 = time.perf_counter()
    #S&R augmentation: 288 -> 864 trials
    eegnet_aug_acc, _, eegnet_aug_cm = EEG_results(aug_bool=True)
    eegnet_aug_runtime = time.perf_counter() - t0
    with open(eegnet_aug_cache, 'wb') as f:
        pickle.dump((eegnet_aug_acc, eegnet_aug_cm, eegnet_aug_runtime), f)
    print(f'Wall time: {eegnet_aug_runtime/60:.1f} min   cached -> {eegnet_aug_cache}')

eegnet_aug_acc = np.array(eegnet_aug_acc)

#per-subject comparison against FBCSP and no aug
print('\nPer-subject accuracy (EEGNet+S&R vs EEGNet vs FBCSP):')
for i, (a_aug, a_eeg, a_fb) in enumerate(zip(eegnet_aug_acc, eegnet_acc, fbcsp_acc), 1):
    print(f'  S{i:02d}: +S&R {a_aug:.4f}   no-aug {a_eeg:.4f}   FBCSP {a_fb:.4f}   '
          f'Δ_FBCSP {a_aug-a_fb:+.4f}   Δ_no-aug {a_aug-a_eeg:+.4f}')
print(f'  mean: +S&R {eegnet_aug_acc.mean():.4f}   no-aug {eegnet_acc.mean():.4f}   '
      f'FBCSP {fbcsp_acc.mean():.4f}')

high_idx = [i-1 for i in [1, 3, 7, 8, 9]]
low_idx  = [i-1 for i in [2, 4, 5, 6]]

gain_high = (eegnet_aug_acc[high_idx] - fbcsp_acc[high_idx]).mean()
gain_low  = (eegnet_aug_acc[low_idx]  - fbcsp_acc[low_idx]).mean()
print(f'\nMean Δ over FBCSP, by group:')
print(f'  high {[1,3,7,8,9]}: {gain_high:+.4f}')
print(f'  low  {[2,4,5,6]}  : {gain_low:+.4f}')
print(f'  -> low minus high: {gain_low - gain_high:+.4f}')

#mean change over no aug vs aug
delta_aug_high = (eegnet_aug_acc[high_idx] - eegnet_acc[high_idx]).mean()
delta_aug_low  = (eegnet_aug_acc[low_idx]  - eegnet_acc[low_idx]).mean()
print(f'\nMean Δ over no-aug EEGNet (S&R-specific), by group:')
print(f'  high: {delta_aug_high:+.4f}')
print(f'  low : {delta_aug_low:+.4f}')
verdict = 'low performers benefit more from S&R' if delta_aug_low > delta_aug_high else 'high performers benefit more from S&R'
print(f'  -> low minus high: {delta_aug_low - delta_aug_high:+.4f}   ({verdict})')

#CM
print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], eegnet_aug_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

Loaded cached EEGNet+S&R results (prior run took 4.6 min)

Per-subject accuracy (EEGNet+S&R vs EEGNet vs FBCSP):
  S01: +S&R 0.8438   no-aug 0.8160   FBCSP 0.6944   Δ_FBCSP +0.1493   Δ_no-aug +0.0278
  S02: +S&R 0.5833   no-aug 0.5556   FBCSP 0.5347   Δ_FBCSP +0.0486   Δ_no-aug +0.0278
  S03: +S&R 0.8924   no-aug 0.8750   FBCSP 0.7986   Δ_FBCSP +0.0937   Δ_no-aug +0.0174
  S04: +S&R 0.6076   no-aug 0.6215   FBCSP 0.5868   Δ_FBCSP +0.0208   Δ_no-aug -0.0139
  S05: +S&R 0.7153   no-aug 0.6493   FBCSP 0.5208   Δ_FBCSP +0.1944   Δ_no-aug +0.0660
  S06: +S&R 0.5972   no-aug 0.5625   FBCSP 0.4062   Δ_FBCSP +0.1910   Δ_no-aug +0.0347
  S07: +S&R 0.7778   no-aug 0.6979   FBCSP 0.7847   Δ_FBCSP -0.0069   Δ_no-aug +0.0799
  S08: +S&R 0.8438   no-aug 0.8056   FBCSP 0.7118   Δ_FBCSP +0.1319   Δ_no-aug +0.0382
  S09: +S&R 0.8160   no-aug 0.7778   FBCSP 0.6944   Δ_FBCSP +0.1215   Δ_no-aug +0.0382
  mean: +S&R 0.7419   no-aug 0.7068   FBCSP 0.6370

Mean Δ over FBCSP, by group:
  high [1, 3, 7, 8, 9]:

---
## Experiment 6: ATCNet + S&R augmentation

Same S&R augmentation on ATCNet. Cached to `atcnet_aug.pkl`.

In [ ]:
#flip to True to ignore the pickle cache and retrain ATCNet+S&R.
FORCE_RERUN_ATCNET_AUG = False

atcnet_aug_cache = os.path.join(RESULTS_DIR, 'atcnet_aug.pkl')

if os.path.exists(atcnet_aug_cache) and not FORCE_RERUN_ATCNET_AUG:
    with open(atcnet_aug_cache, 'rb') as f:
        loaded = pickle.load(f)
    atcnet_aug_acc, atcnet_aug_cm = loaded[0], loaded[1]
    atcnet_aug_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {atcnet_aug_runtime/60:.1f} min' if atcnet_aug_runtime else 'prior runtime not recorded'
    print(f'Loaded cached ATCNet+S&R results ({msg})')
else:
    print('Training ATCNet+S&R on all 9 subjects (300 epochs each, 864 trials/subject)...')
    from normal_results import ATCNet_results
    t0 = time.perf_counter()
    atcnet_aug_acc, _, atcnet_aug_cm = ATCNet_results(aug_bool=True)
    atcnet_aug_runtime = time.perf_counter() - t0
    with open(atcnet_aug_cache, 'wb') as f:
        pickle.dump((atcnet_aug_acc, atcnet_aug_cm, atcnet_aug_runtime), f)
    print(f'Wall time: {atcnet_aug_runtime/60:.1f} min   cached -> {atcnet_aug_cache}')

atcnet_aug_acc = np.array(atcnet_aug_acc)

#per-subject comparison against FBCSP and no aug
print('\nPer-subject accuracy (ATCNet+S&R vs ATCNet vs FBCSP):')
for i, (a_aug, a_atc, a_fb) in enumerate(zip(atcnet_aug_acc, atcnet_acc, fbcsp_acc), 1):
    print(f'  S{i:02d}: +S&R {a_aug:.4f}   no-aug {a_atc:.4f}   FBCSP {a_fb:.4f}   '
          f'Δ_FBCSP {a_aug-a_fb:+.4f}   Δ_no-aug {a_aug-a_atc:+.4f}')
print(f'  mean: +S&R {atcnet_aug_acc.mean():.4f}   no-aug {atcnet_acc.mean():.4f}   '
      f'FBCSP {fbcsp_acc.mean():.4f}')

high_idx = [i-1 for i in [1, 3, 7, 8, 9]]
low_idx  = [i-1 for i in [2, 4, 5, 6]]

gain_high = (atcnet_aug_acc[high_idx] - fbcsp_acc[high_idx]).mean()
gain_low  = (atcnet_aug_acc[low_idx]  - fbcsp_acc[low_idx]).mean()
print(f'\nMean Δ over FBCSP, by group:')
print(f'  high {[1,3,7,8,9]}: {gain_high:+.4f}')
print(f'  low  {[2,4,5,6]}  : {gain_low:+.4f}')
print(f'  -> low minus high: {gain_low - gain_high:+.4f}')

#mean change over no aug vs aug
delta_aug_high = (atcnet_aug_acc[high_idx] - atcnet_acc[high_idx]).mean()
delta_aug_low  = (atcnet_aug_acc[low_idx]  - atcnet_acc[low_idx]).mean()
print(f'\nMean Δ over no-aug ATCNet (S&R-specific), by group:')
print(f'  high: {delta_aug_high:+.4f}')
print(f'  low : {delta_aug_low:+.4f}')
verdict = 'low performers benefit more from S&R' if delta_aug_low > delta_aug_high else 'high performers benefit more from S&R'
print(f'  -> low minus high: {delta_aug_low - delta_aug_high:+.4f}   ({verdict})')

#CM
print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], atcnet_aug_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

Loaded cached ATCNet+S&R results (prior run took 30.7 min)

Per-subject accuracy (ATCNet+S&R vs ATCNet vs FBCSP):
  S01: +S&R 0.7188   no-aug 0.7188   FBCSP 0.6944   Δ_FBCSP +0.0243   Δ_no-aug +0.0000
  S02: +S&R 0.5000   no-aug 0.5417   FBCSP 0.5347   Δ_FBCSP -0.0347   Δ_no-aug -0.0417
  S03: +S&R 0.8819   no-aug 0.8438   FBCSP 0.7986   Δ_FBCSP +0.0833   Δ_no-aug +0.0382
  S04: +S&R 0.5625   no-aug 0.6493   FBCSP 0.5868   Δ_FBCSP -0.0243   Δ_no-aug -0.0868
  S05: +S&R 0.6979   no-aug 0.6806   FBCSP 0.5208   Δ_FBCSP +0.1771   Δ_no-aug +0.0174
  S06: +S&R 0.6146   no-aug 0.5868   FBCSP 0.4062   Δ_FBCSP +0.2083   Δ_no-aug +0.0278
  S07: +S&R 0.7535   no-aug 0.6736   FBCSP 0.7847   Δ_FBCSP -0.0313   Δ_no-aug +0.0799
  S08: +S&R 0.7847   no-aug 0.7500   FBCSP 0.7118   Δ_FBCSP +0.0729   Δ_no-aug +0.0347
  S09: +S&R 0.7326   no-aug 0.7465   FBCSP 0.6944   Δ_FBCSP +0.0382   Δ_no-aug -0.0139
  mean: +S&R 0.6941   no-aug 0.6879   FBCSP 0.6370

Mean Δ over FBCSP, by group:
  high [1, 3, 7, 8, 9]

---
## Experiment 7: MI-Mamba + S&R augmentation

S&R on Mamba. It ran below FBCSP without augmentation, so this tests whether tripling the training set fixes that. 500 epochs. Cached to `mamba_aug.pkl`.

In [ ]:
#flip to True to ignore the pickle cache and retrain Mamba+S&R.
FORCE_RERUN_MAMBA_AUG = False

mamba_aug_cache = os.path.join(RESULTS_DIR, 'mamba_aug.pkl')

if os.path.exists(mamba_aug_cache) and not FORCE_RERUN_MAMBA_AUG:
    with open(mamba_aug_cache, 'rb') as f:
        loaded = pickle.load(f)
    mamba_aug_acc, mamba_aug_cm = loaded[0], loaded[1]
    mamba_aug_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {mamba_aug_runtime/60:.1f} min' if mamba_aug_runtime else 'prior runtime not recorded'
    print(f'Loaded cached Mamba+S&R results ({msg})')
else:
    print('Training MI-Mamba+S&R on all 9 subjects (500 epochs each, 864 trials/subject)...')
    from normal_results import Mamba_results
    t0 = time.perf_counter()
    mamba_aug_acc, _, mamba_aug_cm = Mamba_results(aug_bool=True)
    mamba_aug_runtime = time.perf_counter() - t0
    with open(mamba_aug_cache, 'wb') as f:
        pickle.dump((mamba_aug_acc, mamba_aug_cm, mamba_aug_runtime), f)
    print(f'Wall time: {mamba_aug_runtime/60:.1f} min   cached -> {mamba_aug_cache}')

mamba_aug_acc = np.array(mamba_aug_acc)

#per-subject comparison against FBCSP and no aug
print('\nPer-subject accuracy (Mamba+S&R vs Mamba vs FBCSP):')
for i, (a_aug, a_m, a_fb) in enumerate(zip(mamba_aug_acc, mamba_acc, fbcsp_acc), 1):
    print(f'  S{i:02d}: +S&R {a_aug:.4f}   no-aug {a_m:.4f}   FBCSP {a_fb:.4f}   '
          f'Δ_FBCSP {a_aug-a_fb:+.4f}   Δ_no-aug {a_aug-a_m:+.4f}')
print(f'  mean: +S&R {mamba_aug_acc.mean():.4f}   no-aug {mamba_acc.mean():.4f}   '
      f'FBCSP {fbcsp_acc.mean():.4f}')

high_idx = [i-1 for i in [1, 3, 7, 8, 9]]
low_idx  = [i-1 for i in [2, 4, 5, 6]]

gain_high = (mamba_aug_acc[high_idx] - fbcsp_acc[high_idx]).mean()
gain_low  = (mamba_aug_acc[low_idx]  - fbcsp_acc[low_idx]).mean()
print(f'\nMean Δ over FBCSP, by group:')
print(f'  high {[1,3,7,8,9]}: {gain_high:+.4f}')
print(f'  low  {[2,4,5,6]}  : {gain_low:+.4f}')
print(f'  -> low minus high: {gain_low - gain_high:+.4f}')

#mean change over no aug vs aug
delta_aug_high = (mamba_aug_acc[high_idx] - mamba_acc[high_idx]).mean()
delta_aug_low  = (mamba_aug_acc[low_idx]  - mamba_acc[low_idx]).mean()
print(f'\nMean Δ over no-aug Mamba (S&R-specific), by group:')
print(f'  high: {delta_aug_high:+.4f}')
print(f'  low : {delta_aug_low:+.4f}')
verdict = 'low performers benefit more from S&R' if delta_aug_low > delta_aug_high else 'high performers benefit more from S&R'
print(f'  -> low minus high: {delta_aug_low - delta_aug_high:+.4f}   ({verdict})')

#CM
print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], mamba_aug_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

Loaded cached Mamba+S&R results (prior run took 9.5 min)

Per-subject accuracy (Mamba+S&R vs Mamba vs FBCSP):
  S01: +S&R 0.6424   no-aug 0.6007   FBCSP 0.6944   Δ_FBCSP -0.0521   Δ_no-aug +0.0417
  S02: +S&R 0.3542   no-aug 0.3611   FBCSP 0.5347   Δ_FBCSP -0.1806   Δ_no-aug -0.0069
  S03: +S&R 0.6319   no-aug 0.7326   FBCSP 0.7986   Δ_FBCSP -0.1667   Δ_no-aug -0.1007
  S04: +S&R 0.4514   no-aug 0.4375   FBCSP 0.5868   Δ_FBCSP -0.1354   Δ_no-aug +0.0139
  S05: +S&R 0.3472   no-aug 0.2847   FBCSP 0.5208   Δ_FBCSP -0.1736   Δ_no-aug +0.0625
  S06: +S&R 0.3299   no-aug 0.3472   FBCSP 0.4062   Δ_FBCSP -0.0764   Δ_no-aug -0.0174
  S07: +S&R 0.5625   no-aug 0.5069   FBCSP 0.7847   Δ_FBCSP -0.2222   Δ_no-aug +0.0556
  S08: +S&R 0.6944   no-aug 0.6736   FBCSP 0.7118   Δ_FBCSP -0.0174   Δ_no-aug +0.0208
  S09: +S&R 0.6007   no-aug 0.6250   FBCSP 0.6944   Δ_FBCSP -0.0938   Δ_no-aug -0.0243
  mean: +S&R 0.5127   no-aug 0.5077   FBCSP 0.6370

Mean Δ over FBCSP, by group:
  high [1, 3, 7, 8, 9]: -0

---
## Experiment 8: BYOL pretrain (no augmentation)

For each target subject, pretrain EEGNet on the other 8 subjects’ data using BYOL (self-supervised, no labels), then finetune on the target’s 288 labeled trials. Does cross-subject pretraining help?

NOTE: this runs slow, 9 pretrains + 9 finetunes, is about 30–60 min on A100. Cached to `byol_noaug.pkl`.

In [1]:
#flip to True to ignore the pickle cache and re-run BYOL no-aug.
FORCE_RERUN_BYOL = False

byol_cache = os.path.join(RESULTS_DIR, 'byol_noaug.pkl')

if os.path.exists(byol_cache) and not FORCE_RERUN_BYOL:
    with open(byol_cache, 'rb') as f:
        loaded = pickle.load(f)
    byol_acc, byol_cm = loaded[0], loaded[1]
    byol_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {byol_runtime/60:.1f} min' if byol_runtime else 'prior runtime not recorded'
    print(f'Loaded cached BYOL no-aug results ({msg})')
else:
    print('Running BYOL pretrain LOSO + finetune (no aug) on all 9 subjects...')
    print('  9x [300 epochs pretrain on ~2300 trials, then 300 epochs finetune on 288 trials]')
    from normal_results import SSL_results
    t0 = time.perf_counter()
    # finetune on raw 288 trials (no augmentation)
    byol_acc, _, byol_cm = SSL_results(aug_bool=False)
    byol_runtime = time.perf_counter() - t0
    with open(byol_cache, 'wb') as f:
        pickle.dump((byol_acc, byol_cm, byol_runtime), f)
    print(f'Wall time: {byol_runtime/60:.1f} min   cached -> {byol_cache}')

byol_acc = np.array(byol_acc)

#BYOL vs no-SSL EEGNet vs FBCSP per subject
print('\nPer-subject accuracy (BYOL vs EEGNet vs FBCSP):')
for i, (a_b, a_eeg, a_fb) in enumerate(zip(byol_acc, eegnet_acc, fbcsp_acc), 1):
    print(f'  S{i:02d}: BYOL {a_b:.4f}   EEGNet {a_eeg:.4f}   FBCSP {a_fb:.4f}   '
          f'Δ_FBCSP {a_b-a_fb:+.4f}   Δ_EEGNet {a_b-a_eeg:+.4f}')
print(f'  mean: BYOL {byol_acc.mean():.4f}   EEGNet {eegnet_acc.mean():.4f}   '
      f'FBCSP {fbcsp_acc.mean():.4f}')

high_idx = [i-1 for i in [1, 3, 7, 8, 9]]
low_idx  = [i-1 for i in [2, 4, 5, 6]]

gain_high = (byol_acc[high_idx] - fbcsp_acc[high_idx]).mean()
gain_low  = (byol_acc[low_idx]  - fbcsp_acc[low_idx]).mean()
print(f'\nMean Δ over FBCSP, by group:')
print(f'  high {[1,3,7,8,9]}: {gain_high:+.4f}')
print(f'  low  {[2,4,5,6]}  : {gain_low:+.4f}')
print(f'  -> low minus high: {gain_low - gain_high:+.4f}')

#mean change over BYOL vs plain EEGNet
delta_byol_high = (byol_acc[high_idx] - eegnet_acc[high_idx]).mean()
delta_byol_low  = (byol_acc[low_idx]  - eegnet_acc[low_idx]).mean()
print(f'\nMean Δ over no-SSL EEGNet (BYOL-specific), by group:')
print(f'  high: {delta_byol_high:+.4f}')
print(f'  low : {delta_byol_low:+.4f}')
verdict = 'low performers benefit more from BYOL' if delta_byol_low > delta_byol_high else 'high performers benefit more from BYOL'
print(f'  -> low minus high: {delta_byol_low - delta_byol_high:+.4f}   ({verdict})')

#CM
print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], byol_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

NameError: name 'os' is not defined

---
## Experiment 9: BYOL pretrain + S&R finetune

Same BYOL pretraining, but finetune with S&R augmentation (864 trials instead of 288). Do SSL and augmentation stack?

Same runtime as Experiment 8. Cached to `byol_aug.pkl`.

In [ ]:
#flip to True to ignore the pickle cache and re-run BYOL+S&R.
FORCE_RERUN_BYOL_AUG = False

byol_aug_cache = os.path.join(RESULTS_DIR, 'byol_aug.pkl')

if os.path.exists(byol_aug_cache) and not FORCE_RERUN_BYOL_AUG:
    with open(byol_aug_cache, 'rb') as f:
        loaded = pickle.load(f)
    byol_aug_acc, byol_aug_cm = loaded[0], loaded[1]
    byol_aug_runtime = loaded[2] if len(loaded) > 2 else None
    msg = f'prior run took {byol_aug_runtime/60:.1f} min' if byol_aug_runtime else 'prior runtime not recorded'
    print(f'Loaded cached BYOL+S&R results ({msg})')
else:
    print('Running BYOL pretrain LOSO + finetune+S&R on all 9 subjects...')
    print('  9x [300 epochs pretrain, then 300 epochs finetune on 864 trials]')
    from normal_results import SSL_results
    t0 = time.perf_counter()
    byol_aug_acc, _, byol_aug_cm = SSL_results(aug_bool=True)
    byol_aug_runtime = time.perf_counter() - t0
    with open(byol_aug_cache, 'wb') as f:
        pickle.dump((byol_aug_acc, byol_aug_cm, byol_aug_runtime), f)
    print(f'Wall time: {byol_aug_runtime/60:.1f} min   cached -> {byol_aug_cache}')

byol_aug_acc = np.array(byol_aug_acc)

#BYOL+S&R vs BYOL vs EEGNet+S&R vs FBCSP per subject
print('\nPer-subject accuracy (BYOL+S&R vs BYOL vs FBCSP):')
for i, (a_ba, a_b, a_fb) in enumerate(zip(byol_aug_acc, byol_acc, fbcsp_acc), 1):
    print(f'  S{i:02d}: BYOL+S&R {a_ba:.4f}   BYOL {a_b:.4f}   FBCSP {a_fb:.4f}   '
          f'Δ_FBCSP {a_ba-a_fb:+.4f}   Δ_BYOL {a_ba-a_b:+.4f}')
print(f'  mean: BYOL+S&R {byol_aug_acc.mean():.4f}   BYOL {byol_acc.mean():.4f}   '
      f'FBCSP {fbcsp_acc.mean():.4f}')

high_idx = [i-1 for i in [1, 3, 7, 8, 9]]
low_idx  = [i-1 for i in [2, 4, 5, 6]]

gain_high = (byol_aug_acc[high_idx] - fbcsp_acc[high_idx]).mean()
gain_low  = (byol_aug_acc[low_idx]  - fbcsp_acc[low_idx]).mean()
print(f'\nMean Δ over FBCSP, by group:')
print(f'  high {[1,3,7,8,9]}: {gain_high:+.4f}')
print(f'  low  {[2,4,5,6]}  : {gain_low:+.4f}')
print(f'  -> low minus high: {gain_low - gain_high:+.4f}')

#what's the benefit of S&R on top of BYOL?
delta_aug_high = (byol_aug_acc[high_idx] - byol_acc[high_idx]).mean()
delta_aug_low  = (byol_aug_acc[low_idx]  - byol_acc[low_idx]).mean()
print(f'\nMean Δ over no-aug BYOL (S&R-on-top-of-SSL), by group:')
print(f'  high: {delta_aug_high:+.4f}')
print(f'  low : {delta_aug_low:+.4f}')
verdict = 'S&R still helps low more on top of SSL' if delta_aug_low > delta_aug_high else 'S&R helps high more on top of SSL'
print(f'  -> low minus high: {delta_aug_low - delta_aug_high:+.4f}   ({verdict})')

#CM
print('\nConfusion matrix (rows=true, cols=pred):')
print('              Left   Right  Feet   Tongue')
for name, row in zip(['Left  ', 'Right ', 'Feet  ', 'Tongue'], byol_aug_cm):
    print(f'  true {name}  ', '  '.join(f'{v:5d}' for v in row))

---
## Experiment 10: BYOL ablation (leave-one-out)

For each low-performer target {2, 4, 5, 6}, retrain BYOL pretraining 8 times leaving one source subject out each time. Measures how much each source subject actually contributed to the target’s accuracy.

4 targets × 8 exclusions = 32 runs, takes a few hours. Checkpointed after each run so it’s safe to stop and resume.

In [ ]:
# Point ablation.py at our results dir (env vars consumed by the patched run_ablation).
os.environ['ABLATION_CHECKPOINT_DIR'] = os.path.join(RESULTS_DIR, 'ablation_checkpoints')
os.environ['ABLATION_RESULTS_PATH']   = os.path.join(RESULTS_DIR, 'ABLATION_RESULTS.txt')

#uses exp 9 (BYOL+S&R) accuracies as the per-target baseline
ablation_baselines = {i: float(byol_aug_acc[i-1]) for i in [2, 4, 5, 6]}
print('Ablation baselines (from exp 9 BYOL+S&R):')
for k, v in ablation_baselines.items():
    print(f'  S{k:02d}: {v:.4f}')

from ablation import run_ablation

t0 = time.perf_counter()

#creates checkpointing for re-running this cell
ablation_results = run_ablation(
    subjectData, subjectDataEVAL, DATA_DIR,
    baselines=ablation_baselines,
)
ablation_runtime = time.perf_counter() - t0
print(f'\nAblation wall time this run: {ablation_runtime/60:.1f} min')

#per-target tables + per-target high/low donor split
LOW_TARGETS = [2, 4, 5, 6]
HIGH_DONORS = {1, 3, 7, 8, 9}
LOW_DONORS  = {2, 4, 5, 6}

agg_drops_when_excluding_high = []
agg_drops_when_excluding_low  = []

for target in LOW_TARGETS:
    runs = ablation_results.get(target, {})
    baseline = ablation_baselines[target]
    print(f'\nTarget S{target:02d} (baseline SSL+aug: {baseline:.4f}):')
    print(f'  donor   type   acc       drop')
    print(f'  -----   ----   -------   -------')

    drops_high, drops_low = [], []
    for donor in sorted(runs.keys()):
        acc  = runs[donor]
        drop = baseline - acc
        donor_type = 'high' if donor in HIGH_DONORS else 'low '
        print(f'  S{donor:02d}     {donor_type}   {acc:.4f}    {drop:+.4f}')
        if donor in HIGH_DONORS:
            drops_high.append(drop)
            agg_drops_when_excluding_high.append(drop)
        else:
            drops_low.append(drop)
            agg_drops_when_excluding_low.append(drop)

    if drops_high:
        print(f'  Mean drop excluding HIGH donors: {np.mean(drops_high):+.4f}')
    if drops_low:
        print(f'  Mean drop excluding LOW  donors: {np.mean(drops_low):+.4f}')

#aggregates across all of the low performers
print('\n' + '='*60)
print('Aggregated across all 4 low targets:')
if agg_drops_when_excluding_high and agg_drops_when_excluding_low:
    mean_drop_high = float(np.mean(agg_drops_when_excluding_high))
    mean_drop_low  = float(np.mean(agg_drops_when_excluding_low))
    print(f'  Mean drop when excluding HIGH donors: {mean_drop_high:+.4f}  (n={len(agg_drops_when_excluding_high)})')
    print(f'  Mean drop when excluding LOW  donors: {mean_drop_low:+.4f}  (n={len(agg_drops_when_excluding_low)})')
    diff = mean_drop_high - mean_drop_low
    if diff > 0:
        print(f'  -> high - low: {diff:+.4f}   HIGH donors contribute MORE — supports RQ4')
    else:
        print(f'  -> high - low: {diff:+.4f}   LOW donors contribute more — counter to RQ4')
else:
    print('  (incomplete — keep re-running this cell to finish the 32 ablation runs)')

---
## Paper figures

Generates the four plots used in the paper.

In [ ]:
from plot import plot_all_confusion_matrices, run_erd_analysis, run_ablation_plot

#confusion matrices, no augmentation (FBCSP, EEGNet, Mamba, ATCNet)
plot_all_confusion_matrices(
    {'FBCSP': fbcsp_cm, 'EEGNet': eegnet_cm, 'Mamba': mamba_cm, 'ATCNet': atcnet_cm},
    aug_bool=False
)

#confusion matrices with S&R augmentation (EEGNet, Mamba, ATCNet)
plot_all_confusion_matrices(
    {'EEGNet': eegnet_aug_cm, 'Mamba': mamba_aug_cm, 'ATCNet': atcnet_aug_cm},
    aug_bool=True
)

#ERD topomap
run_erd_analysis(subjectData)

#BYOL ablation bar chart
run_ablation_plot(ablation_results, ablation_baselines)
